[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rsasaki0109/slam-handbook-python/blob/main/part2_practice/ch09_radar_slam/01_radar_odometry.ipynb)

# 第9章: レーダーSLAM - レーダーオドメトリの基礎

**SLAM Handbook Chapter 9, Section 9.1-9.3**

このノートブックでは、レーダーSLAMの基礎を学びます。

## 学習内容
1. レーダーセンサーの基本原理とドップラー速度推定
2. レーダー極座標スキャンからデカルト座標への変換
3. 相関スキャンマッチング（Correlative Scan Matching）
4. LiDARとの比較

レーダーは悪天候（雨、霧、粉塵）に強く、長距離計測が可能なため、自動運転やロボティクスで注目されています。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

## 9.1 ドップラー速度推定

**SLAM Handbook Section 9.1**

レーダーはドップラー効果を利用して、ターゲットの視線方向（radial）速度を直接計測できます。

$$v_r = \frac{f_d \cdot \lambda}{2}$$

ここで：
- $v_r$: 視線方向速度（radial velocity）
- $f_d$: ドップラー周波数シフト
- $\lambda$: レーダー波長

複数のターゲットからのドップラー速度を用いて、センサー自身の速度を推定できます（エゴモーション推定）。

$$v_r^{(i)} = \mathbf{d}_i^T \mathbf{v}_{ego}$$

ここで $\mathbf{d}_i$ はターゲット $i$ への単位方向ベクトルです。

In [ ]:
def estimate_ego_velocity(target_angles, radial_velocities):
    """
    Estimate ego velocity from multiple Doppler measurements.
    v_r^(i) = d_i^T @ v_ego  =>  solve least squares
    """
    # Direction vectors for each target
    D = np.column_stack([np.cos(target_angles), np.sin(target_angles)])
    # Least squares: D @ v_ego = v_r
    v_ego, residuals, _, _ = np.linalg.lstsq(D, radial_velocities, rcond=None)
    return v_ego

# Simulate: sensor moving with true velocity
v_true = np.array([2.0, 1.0])  # [vx, vy] m/s
n_targets = 20

# Random target directions
angles = np.linspace(0, 2 * np.pi, n_targets, endpoint=False)
angles += np.random.uniform(-0.1, 0.1, n_targets)

# Radial velocity = projection of ego velocity onto target direction + noise
noise_std = 0.1
v_radial = np.array([np.dot([np.cos(a), np.sin(a)], v_true) for a in angles])
v_radial_noisy = v_radial + np.random.normal(0, noise_std, n_targets)

# Estimate ego velocity
v_est = estimate_ego_velocity(angles, v_radial_noisy)

print(f"True ego velocity:      [{v_true[0]:.3f}, {v_true[1]:.3f}] m/s")
print(f"Estimated ego velocity: [{v_est[0]:.3f}, {v_est[1]:.3f}] m/s")
print(f"Error norm:             {np.linalg.norm(v_true - v_est):.4f} m/s")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Doppler measurements
ax = axes[0]
for i in range(n_targets):
    color = 'red' if v_radial_noisy[i] > 0 else 'blue'
    ax.arrow(0, 0, np.cos(angles[i]) * abs(v_radial_noisy[i]),
             np.sin(angles[i]) * abs(v_radial_noisy[i]),
             head_width=0.05, head_length=0.03, fc=color, ec=color, alpha=0.5)
ax.arrow(0, 0, v_true[0], v_true[1], head_width=0.08, head_length=0.05,
         fc='green', ec='green', linewidth=2, label='True velocity')
ax.arrow(0, 0, v_est[0], v_est[1], head_width=0.08, head_length=0.05,
         fc='orange', ec='orange', linewidth=2, label='Estimated velocity')
ax.set_xlim(-3, 3)
ax.set_ylim(-3, 3)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.legend()
ax.set_title('Doppler Velocity Estimation')
ax.set_xlabel('x [m/s]')
ax.set_ylabel('y [m/s]')

# Plot 2: Radial velocity vs angle
ax = axes[1]
ax.scatter(np.degrees(angles), v_radial, label='True radial velocity', s=30)
ax.scatter(np.degrees(angles), v_radial_noisy, label='Noisy measurement', s=30, marker='x')
ax.set_xlabel('Target angle [deg]')
ax.set_ylabel('Radial velocity [m/s]')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_title('Radial Velocity vs Target Angle')

plt.tight_layout()
plt.show()

## 9.2 レーダー極座標スキャンからデカルト座標への変換

**SLAM Handbook Section 9.2**

レーダースキャンは極座標（距離 $r$、方位角 $\theta$）で取得されます。SLAM処理のためにデカルト座標に変換します。

$$x = r \cos\theta, \quad y = r \sin\theta$$

レーダーの特徴：
- **低角度分解能**: LiDARと比較して角度分解能が低い
- **マルチパス反射**: 壁や地面からの多重反射
- **スペックルノイズ**: レーダー特有のノイズパターン

In [ ]:
def generate_radar_scan(obstacles, n_azimuths=400, range_max=50.0, range_res=0.1,
                        noise_std=0.3, speckle_prob=0.05):
    """
    Simulate a radar polar scan from a set of obstacle points.
    Returns power image in polar coordinates (azimuth x range_bins).
    """
    n_range_bins = int(range_max / range_res)
    azimuths = np.linspace(0, 2 * np.pi, n_azimuths, endpoint=False)
    polar_scan = np.zeros((n_azimuths, n_range_bins))

    for obs in obstacles:
        r = np.sqrt(obs[0]**2 + obs[1]**2)
        theta = np.arctan2(obs[1], obs[0]) % (2 * np.pi)
        if r >= range_max:
            continue
        # Find nearest azimuth bin
        az_idx = np.argmin(np.abs(azimuths - theta))
        r_idx = int(r / range_res)
        if 0 <= r_idx < n_range_bins:
            # Spread power across neighboring bins (simulate beam width)
            for da in range(-3, 4):
                for dr in range(-2, 3):
                    ai = (az_idx + da) % n_azimuths
                    ri = r_idx + dr
                    if 0 <= ri < n_range_bins:
                        power = np.exp(-0.5 * (da**2 / 2.0 + dr**2 / 1.0))
                        polar_scan[ai, ri] += power

    # Add noise and speckle
    polar_scan += np.abs(np.random.normal(0, noise_std, polar_scan.shape))
    speckle_mask = np.random.random(polar_scan.shape) < speckle_prob
    polar_scan[speckle_mask] += np.random.uniform(0.5, 2.0, np.sum(speckle_mask))

    return polar_scan, azimuths

def polar_to_cartesian(polar_scan, azimuths, range_res=0.1, threshold=1.5):
    """Convert polar radar scan to Cartesian point cloud."""
    n_range_bins = polar_scan.shape[1]
    ranges = np.arange(n_range_bins) * range_res
    points = []
    for i, az in enumerate(azimuths):
        # Find peaks above threshold
        strong_bins = np.where(polar_scan[i] > threshold)[0]
        for rb in strong_bins:
            r = ranges[rb]
            x = r * np.cos(az)
            y = r * np.sin(az)
            points.append([x, y, polar_scan[i, rb]])
    return np.array(points) if points else np.empty((0, 3))

# Create simulated environment: walls forming an L-shaped corridor
obstacles = []
for x in np.linspace(-20, 20, 200):
    obstacles.append([x, 10])   # top wall
    obstacles.append([x, -10])  # bottom wall
for y in np.linspace(-10, 10, 100):
    obstacles.append([20, y])   # right wall
    obstacles.append([-20, y])  # left wall
# Add some random objects
for _ in range(15):
    obstacles.append([np.random.uniform(-15, 15), np.random.uniform(-8, 8)])
obstacles = np.array(obstacles)

# Generate radar scan
polar_scan, azimuths = generate_radar_scan(obstacles)
cart_points = polar_to_cartesian(polar_scan, azimuths)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Polar scan image
ax = axes[0]
ax.imshow(polar_scan.T, aspect='auto', origin='lower', cmap='hot',
          extent=[0, 360, 0, 50])
ax.set_xlabel('Azimuth [deg]')
ax.set_ylabel('Range [m]')
ax.set_title('Radar Polar Scan (Power Image)')

# Cartesian point cloud
ax = axes[1]
if len(cart_points) > 0:
    sc = ax.scatter(cart_points[:, 0], cart_points[:, 1], c=cart_points[:, 2],
                    s=1, cmap='hot', alpha=0.5)
    plt.colorbar(sc, ax=ax, label='Power')
ax.scatter(obstacles[:, 0], obstacles[:, 1], c='blue', s=5, alpha=0.3, label='True obstacles')
ax.set_xlim(-25, 25)
ax.set_ylim(-15, 15)
ax.set_aspect('equal')
ax.legend()
ax.set_title('Cartesian Point Cloud')
ax.set_xlabel('x [m]')
ax.set_ylabel('y [m]')

# Ground truth
ax = axes[2]
ax.scatter(obstacles[:, 0], obstacles[:, 1], c='blue', s=10)
ax.plot(0, 0, 'r^', markersize=10, label='Radar sensor')
ax.set_xlim(-25, 25)
ax.set_ylim(-15, 15)
ax.set_aspect('equal')
ax.legend()
ax.set_title('Ground Truth Environment')
ax.set_xlabel('x [m]')
ax.set_ylabel('y [m]')

plt.tight_layout()
plt.show()

## 9.3 相関スキャンマッチング

**SLAM Handbook Section 9.3**

相関スキャンマッチング（Correlative Scan Matching）は、2つのレーダースキャン間の相対変位を推定する手法です。LiDARのICP（Iterative Closest Point）に相当しますが、レーダーのパワー画像に対して相互相関を計算します。

スコア関数：
$$S(\Delta x, \Delta y, \Delta\theta) = \sum_{i} P_2(T(\Delta x, \Delta y, \Delta\theta) \cdot \mathbf{p}_i)$$

ここで $P_2$ はスキャン2のパワー値、$\mathbf{p}_i$ はスキャン1の点です。

In [ ]:
def create_occupancy_grid(points, grid_res=0.5, grid_size=80):
    """Create a 2D occupancy grid from point cloud for scan matching."""
    grid = np.zeros((grid_size, grid_size))
    half = grid_size // 2
    for p in points:
        gx = int(p[0] / grid_res) + half
        gy = int(p[1] / grid_res) + half
        if 0 <= gx < grid_size and 0 <= gy < grid_size:
            grid[gy, gx] = max(grid[gy, gx], p[2] if len(p) > 2 else 1.0)
    # Gaussian blur for smooth matching
    from scipy.ndimage import gaussian_filter
    grid = gaussian_filter(grid, sigma=1.5)
    return grid

def correlative_scan_match(grid1_points, grid2, grid_res=0.5, grid_size=80,
                           search_range_xy=5.0, search_range_theta=0.3,
                           step_xy=0.5, step_theta=0.05):
    """Brute-force correlative scan matching."""
    half = grid_size // 2
    best_score = -np.inf
    best_pose = (0, 0, 0)

    dx_range = np.arange(-search_range_xy, search_range_xy + step_xy, step_xy)
    dy_range = np.arange(-search_range_xy, search_range_xy + step_xy, step_xy)
    dtheta_range = np.arange(-search_range_theta, search_range_theta + step_theta, step_theta)

    scores = np.zeros((len(dx_range), len(dy_range)))

    for it, dtheta in enumerate(dtheta_range):
        cos_t, sin_t = np.cos(dtheta), np.sin(dtheta)
        for ix, dx in enumerate(dx_range):
            for iy, dy in enumerate(dy_range):
                score = 0
                for p in grid1_points:
                    # Transform point
                    px = cos_t * p[0] - sin_t * p[1] + dx
                    py = sin_t * p[0] + cos_t * p[1] + dy
                    gx = int(px / grid_res) + half
                    gy = int(py / grid_res) + half
                    if 0 <= gx < grid_size and 0 <= gy < grid_size:
                        score += grid2[gy, gx]
                if score > best_score:
                    best_score = score
                    best_pose = (dx, dy, dtheta)
                if abs(dtheta) < step_theta:
                    scores[ix, iy] = score

    return best_pose, best_score, scores, dx_range, dy_range

# Generate two scans with known transformation
true_dx, true_dy, true_dtheta = 2.0, 1.0, 0.1  # True displacement

# Scan 1: original position
scan1_points = cart_points[:, :2].copy() if len(cart_points) > 0 else np.empty((0, 2))

# Scan 2: transform obstacles and regenerate
cos_t = np.cos(-true_dtheta)
sin_t = np.sin(-true_dtheta)
obstacles_2 = obstacles.copy()
obstacles_2[:, 0] = cos_t * (obstacles[:, 0] - true_dx) - sin_t * (obstacles[:, 1] - true_dy)
obstacles_2[:, 1] = sin_t * (obstacles[:, 0] - true_dx) + cos_t * (obstacles[:, 1] - true_dy)

polar_scan2, azimuths2 = generate_radar_scan(obstacles_2)
cart_points2 = polar_to_cartesian(polar_scan2, azimuths2)

# Create grid for scan 2
grid2 = create_occupancy_grid(cart_points2)

# Subsample scan1 points for speed
step = max(1, len(cart_points) // 200)
scan1_sub = cart_points[::step]

# Run correlative scan matching
est_pose, score, score_map, dx_r, dy_r = correlative_scan_match(
    scan1_sub, grid2, search_range_xy=4.0, step_xy=0.25, step_theta=0.02)

print(f"True transform:      dx={true_dx:.2f}, dy={true_dy:.2f}, dtheta={np.degrees(true_dtheta):.1f} deg")
print(f"Estimated transform: dx={est_pose[0]:.2f}, dy={est_pose[1]:.2f}, dtheta={np.degrees(est_pose[2]):.1f} deg")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Score map
ax = axes[0]
im = ax.imshow(score_map.T, origin='lower', extent=[dx_r[0], dx_r[-1], dy_r[0], dy_r[-1]],
               cmap='viridis', aspect='equal')
ax.plot(true_dx, true_dy, 'r*', markersize=15, label='True')
ax.plot(est_pose[0], est_pose[1], 'wx', markersize=12, markeredgewidth=2, label='Estimated')
plt.colorbar(im, ax=ax, label='Correlation Score')
ax.set_xlabel('dx [m]')
ax.set_ylabel('dy [m]')
ax.set_title('Correlative Scan Matching Score Map')
ax.legend()

# Scan alignment result
ax = axes[1]
if len(cart_points2) > 0:
    ax.scatter(cart_points2[:, 0], cart_points2[:, 1], s=1, alpha=0.3, c='blue', label='Scan 2')
if len(cart_points) > 0:
    cos_e, sin_e = np.cos(est_pose[2]), np.sin(est_pose[2])
    aligned = cart_points[:, :2].copy()
    aligned_x = cos_e * aligned[:, 0] - sin_e * aligned[:, 1] + est_pose[0]
    aligned_y = sin_e * aligned[:, 0] + cos_e * aligned[:, 1] + est_pose[1]
    ax.scatter(aligned_x, aligned_y, s=1, alpha=0.3, c='red', label='Scan 1 (aligned)')
ax.set_xlim(-25, 25)
ax.set_ylim(-15, 15)
ax.set_aspect('equal')
ax.legend()
ax.set_title('Scan Alignment Result')
ax.set_xlabel('x [m]')
ax.set_ylabel('y [m]')

plt.tight_layout()
plt.show()

## 9.4 LiDARとの比較

レーダーとLiDARの主な違いを比較します。

| 特性 | レーダー | LiDAR |
|------|---------|-------|
| 角度分解能 | 低（~1-2度） | 高（~0.1度） |
| 距離分解能 | 中（~0.1m） | 高（~0.01m） |
| 最大距離 | 長（~200m+） | 中（~100m） |
| 悪天候耐性 | 強い | 弱い |
| 速度計測 | 直接（ドップラー） | 間接（フレーム間差分） |
| コスト | 中～低 | 高 |

In [ ]:
# Comparison: Radar vs LiDAR scan simulation
def generate_lidar_scan(obstacles, n_beams=720, range_max=50.0, noise_std=0.02):
    """Simulate a LiDAR scan (much higher resolution, lower noise)."""
    angles = np.linspace(0, 2 * np.pi, n_beams, endpoint=False)
    ranges = np.full(n_beams, range_max)

    for obs in obstacles:
        r = np.sqrt(obs[0]**2 + obs[1]**2)
        theta = np.arctan2(obs[1], obs[0]) % (2 * np.pi)
        beam_idx = np.argmin(np.abs(angles - theta))
        if r < ranges[beam_idx]:
            ranges[beam_idx] = r + np.random.normal(0, noise_std)

    points_x = ranges * np.cos(angles)
    points_y = ranges * np.sin(angles)
    valid = ranges < range_max * 0.99
    return points_x[valid], points_y[valid]

lidar_x, lidar_y = generate_lidar_scan(obstacles)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Radar
ax = axes[0]
if len(cart_points) > 0:
    ax.scatter(cart_points[:, 0], cart_points[:, 1], s=1, alpha=0.3, c='red')
ax.plot(0, 0, 'k^', markersize=10)
ax.set_xlim(-25, 25)
ax.set_ylim(-15, 15)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.set_title(f'Radar Scan ({len(cart_points)} points)\n低分解能・スペックルノイズあり')
ax.set_xlabel('x [m]')
ax.set_ylabel('y [m]')

# LiDAR
ax = axes[1]
ax.scatter(lidar_x, lidar_y, s=1, alpha=0.5, c='green')
ax.plot(0, 0, 'k^', markersize=10)
ax.set_xlim(-25, 25)
ax.set_ylim(-15, 15)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.set_title(f'LiDAR Scan ({len(lidar_x)} points)\n高分解能・低ノイズ')
ax.set_xlabel('x [m]')
ax.set_ylabel('y [m]')

plt.tight_layout()
plt.show()

## 演習問題

1. **ドップラー推定のロバスト化**: `estimate_ego_velocity` 関数にRANSACを追加し、外れ値（動的物体）に頑健な速度推定を実装せよ。
2. **スキャンマッチングの精度向上**: `correlative_scan_match` の探索ステップを粗→密の2段階にして、計算速度と精度を両立させよ。
3. **ノイズの影響分析**: レーダーのノイズ標準偏差 `noise_std` を変化させたとき、スキャンマッチングの精度がどう変化するか調べよ。

## まとめ

本ノートブックでは以下を学びました：

1. **ドップラー速度推定**: レーダーのドップラー効果を利用して、最小二乗法でエゴモーション（自己速度）を推定する方法
2. **極座標→デカルト変換**: レーダーの極座標パワー画像をデカルト座標の点群に変換する処理
3. **相関スキャンマッチング**: パワー画像の相互相関に基づくスキャン間の相対姿勢推定
4. **LiDARとの比較**: レーダーは分解能で劣るが、悪天候耐性と速度計測で優位

レーダーSLAMは、特に自動車の自動運転において、LiDARやカメラと相補的なセンサーとして重要性が増しています。

**参考**: SLAM Handbook Chapter 9